# Phase 3 — Shared Model-Facing Preprocessing and Interface Validation

This notebook performs model-facing preprocessing for the finance ABSA project.

Goals of this phase:

1. load the Phase 2 exported files for ATE, ASC, and canonical headline-level data;
2. initialize a shared DeBERTa tokenizer and common label mappings;
3. convert ATE data into tokenized sequence-labeling inputs with subword-aligned labels;
4. convert ASC data into tokenized sentence-aspect classification inputs;
5. define a shared prediction schema for ATE, ASC, and later merge/pipeline integration;
6. run smoke tests to verify that the three branches are interface-compatible.

This notebook does NOT perform full training.
It only validates I/O, preprocessing logic, and model-facing data consistency.

In [1]:
import json
import re
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

d:\anaconda3\envs\tpwng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("outputs_phase2")

ATE_TRAIN_PATH = DATA_DIR / "ate_train.jsonl"
ATE_VAL_PATH   = DATA_DIR / "ate_val.jsonl"
ATE_TEST_PATH  = DATA_DIR / "ate_test.jsonl"

ASC_TRAIN_PATH = DATA_DIR / "asc_train.jsonl"
ASC_VAL_PATH   = DATA_DIR / "asc_val.jsonl"
ASC_TEST_PATH  = DATA_DIR / "asc_test.jsonl"

CANONICAL_TRAIN_PATH = DATA_DIR / "train_canonical.jsonl"
CANONICAL_VAL_PATH   = DATA_DIR / "val_canonical.jsonl"
CANONICAL_TEST_PATH  = DATA_DIR / "test_canonical.jsonl"

SPLIT_STATS_PATH = DATA_DIR / "split_stats_report.json"
SPLIT_MANIFEST_PATH = DATA_DIR / "split_manifest.json"

OUT_DIR = Path("outputs_phase3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ATE_SMOKE_REPORT_PATH = OUT_DIR / "ate_smoke_report.json"
ASC_SMOKE_REPORT_PATH = OUT_DIR / "asc_smoke_report.json"
MERGE_SMOKE_REPORT_PATH = OUT_DIR / "merge_smoke_report.json"
PHASE3_CHECKLIST_PATH = OUT_DIR / "phase3_checklist.csv"

## Step 1. Define shared model-facing configuration

This section defines the tokenizer backbone, max sequence lengths, label maps, and shared constants.
The purpose is to ensure that all downstream branches use the same preprocessing rules.

In [3]:
BACKBONE_NAME = "microsoft/deberta-v3-base"
# Optional fallback if needed later:
# BACKBONE_NAME = "microsoft/deberta-v3-small"

MAX_LEN_ATE = 128
MAX_LEN_ASC = 128

ATE_LABEL2ID = {
    "O": 0,
    "B-ASP": 1,
    "I-ASP": 2
}
ATE_ID2LABEL = {v: k for k, v in ATE_LABEL2ID.items()}

ASC_LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}
ASC_ID2LABEL = {v: k for k, v in ASC_LABEL2ID.items()}

IGNORE_INDEX = -100
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Step 2. Load the shared tokenizer

We use one shared tokenizer configuration so that ATE, ASC, and later joint integration remain compatible.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME)
print("Loaded tokenizer:", BACKBONE_NAME)
print("Tokenizer type:", type(tokenizer).__name__)
print("Pad token:", tokenizer.pad_token)
print("CLS token:", tokenizer.cls_token)
print("SEP token:", tokenizer.sep_token)

Loaded tokenizer: microsoft/deberta-v3-base
Tokenizer type: DebertaV2Tokenizer
Pad token: [PAD]
CLS token: [CLS]
SEP token: [SEP]


## Step 3. Define shared JSONL loading utilities

All branches must read data using the same shared loading logic.

In [5]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records

In [6]:
ate_train = load_jsonl(ATE_TRAIN_PATH)
ate_val   = load_jsonl(ATE_VAL_PATH)
ate_test  = load_jsonl(ATE_TEST_PATH)

asc_train = load_jsonl(ASC_TRAIN_PATH)
asc_val   = load_jsonl(ASC_VAL_PATH)
asc_test  = load_jsonl(ASC_TEST_PATH)

canonical_train = load_jsonl(CANONICAL_TRAIN_PATH)
canonical_val   = load_jsonl(CANONICAL_VAL_PATH)
canonical_test  = load_jsonl(CANONICAL_TEST_PATH)

## Step 4. Load split metadata and verify expected counts

This step is not strictly required for preprocessing, but it confirms that the notebook is using the intended Phase 2 exports.

In [7]:
with open(SPLIT_STATS_PATH, "r", encoding="utf-8") as f:
    split_stats = json.load(f)

with open(SPLIT_MANIFEST_PATH, "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

pprint(split_stats)

{'asc_validation': {'test_all_labels_valid': True,
                    'test_records': 1468,
                    'train_all_labels_valid': True,
                    'train_records': 11443,
                    'val_all_labels_valid': True,
                    'val_records': 1438},
 'ate_validation': {'test_rows': 1079,
                    'test_token_label_len_ok': True,
                    'train_rows': 8576,
                    'train_token_label_len_ok': True,
                    'val_rows': 1071,
                    'val_token_label_len_ok': True},
 'leakage_report': {'train_test_overlap': 0,
                    'train_val_overlap': 0,
                    'val_test_overlap': 0},
 'num_model_ready_rows': 10726,
 'num_test_groups': 1066,
 'num_test_rows': 1079,
 'num_train_groups': 8526,
 'num_train_rows': 8576,
 'num_val_groups': 1066,
 'num_val_rows': 1071,
 'review_holdout_rows': 27,
 'review_summary': {'review_holdout_aspects': 27,
                    'review_holdout_headlines': 2

In [8]:
print("ATE train/val/test rows:", len(ate_train), len(ate_val), len(ate_test))
print("ASC train/val/test rows:", len(asc_train), len(asc_val), len(asc_test))
print("Canonical train/val/test rows:", len(canonical_train), len(canonical_val), len(canonical_test))

ATE train/val/test rows: 8576 1071 1079
ASC train/val/test rows: 11443 1438 1468
Canonical train/val/test rows: 8576 1071 1079


## Step 5. Run shared file-level integrity checks

We verify that:
- split files are non-empty,
- core fields exist,
- split counts are internally consistent.

In [9]:
def check_required_fields(records, required_fields):
    if len(records) == 0:
        return False, ["empty_records"]
    missing = []
    sample = records[0]
    for f in required_fields:
        if f not in sample:
            missing.append(f)
    return len(missing) == 0, missing

In [10]:
ate_ok, ate_missing = check_required_fields(
    ate_train, ["doc_id", "raw_id", "split", "text", "tokens", "token_offsets", "bio_labels", "aspects"]
)

asc_ok, asc_missing = check_required_fields(
    asc_train, ["doc_id", "raw_id", "split", "sentence", "aspect", "sentiment", "char_from", "char_to", "match_type"]
)

canon_ok, canon_missing = check_required_fields(
    canonical_train, ["raw_id", "doc_id", "text", "group_key", "num_aspects", "label_signature", "aspects", "split"]
)

print("ATE fields OK:", ate_ok, ate_missing)
print("ASC fields OK:", asc_ok, asc_missing)
print("Canonical fields OK:", canon_ok, canon_missing)

ATE fields OK: True []
ASC fields OK: True []
Canonical fields OK: True []


## Step 6. Inspect one ATE sample

ATE uses word-level tokens and BIO labels as the supervision source.
The next task is to map these word-level labels onto tokenizer subwords.

In [11]:
ate_sample = ate_train[0]
pprint(ate_sample)

{'aspects': [{'char_from': 0,
              'char_to': 8,
              'match_status': 'matched',
              'match_type': 'whole_word_unique',
              'sentiment': 'neutral',
              'term': 'SpiceJet'}],
 'bio_labels': ['B-ASP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 'doc_id': 'sentfin_000001',
 'raw_id': 1,
 'split': 'train',
 'text': 'SpiceJet to issue 6.4 crore warrants to promoters',
 'token_offsets': [[0, 8],
                   [9, 11],
                   [12, 17],
                   [18, 19],
                   [19, 20],
                   [20, 21],
                   [22, 27],
                   [28, 36],
                   [37, 39],
                   [40, 49]],
 'tokens': ['SpiceJet',
            'to',
            'issue',
            '6',
            '.',
            '4',
            'crore',
            'warrants',
            'to',
            'promoters']}


In [12]:
print("Text:", ate_sample["text"])
print("Tokens:", ate_sample["tokens"])
print("BIO labels:", ate_sample["bio_labels"])
print("Lengths:", len(ate_sample["tokens"]), len(ate_sample["bio_labels"]))

Text: SpiceJet to issue 6.4 crore warrants to promoters
Tokens: ['SpiceJet', 'to', 'issue', '6', '.', '4', 'crore', 'warrants', 'to', 'promoters']
BIO labels: ['B-ASP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Lengths: 10 10


## Step 7. Define word-level to subword-level label alignment for ATE

The exported ATE files are currently word-level.
DeBERTa tokenization may split one word into multiple subwords.

Policy used here:
- assign the original BIO label to the first subword of each word,
- assign IGNORE_INDEX to continuation subwords,
- assign IGNORE_INDEX to special tokens and padding.

In [13]:
def encode_ate_example(example, tokenizer, max_length=128):
    tokens = example["tokens"]
    word_labels = example["bio_labels"]

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        padding=False
    )

    word_ids = encoding.word_ids()
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(IGNORE_INDEX)
        elif word_id != prev_word_id:
            aligned_labels.append(ATE_LABEL2ID[word_labels[word_id]])
        else:
            aligned_labels.append(IGNORE_INDEX)
        prev_word_id = word_id

    encoding["labels"] = aligned_labels
    return encoding

In [14]:
ate_encoded = encode_ate_example(ate_sample, tokenizer, max_length=MAX_LEN_ATE)

print("input_ids length:", len(ate_encoded["input_ids"]))
print("attention_mask length:", len(ate_encoded["attention_mask"]))
print("labels length:", len(ate_encoded["labels"]))
print("word_ids:", ate_encoded.word_ids()[:30])
print("labels:", ate_encoded["labels"][:30])

input_ids length: 13
attention_mask length: 13
labels length: 13
word_ids: [None, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, None]
labels: [-100, 1, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]


## Step 8. Sanity-check the aligned ATE labels

We decode token pieces back into readable form and verify that:
- label assignment occurs at the first subword only,
- continuation pieces are ignored,
- special tokens are ignored.

In [15]:
def pretty_print_ate_alignment(example, encoding, tokenizer, max_rows=60):
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    word_ids = encoding.word_ids()
    labels = encoding["labels"]

    rows = []
    for i, (tok, wid, lab) in enumerate(zip(tokens, word_ids, labels)):
        rows.append({
            "idx": i,
            "subword": tok,
            "word_id": wid,
            "label_id": lab,
            "label_name": None if lab == IGNORE_INDEX else ATE_ID2LABEL[lab]
        })

    return pd.DataFrame(rows).head(max_rows)

In [16]:
pretty_print_ate_alignment(ate_sample, ate_encoded, tokenizer)

,idx,subword,word_id,label_id,label_name
0,0,[CLS],NaN,-100,NaN
1,1,▁Spice,0.0,1,B-ASP
2,2,Jet,0.0,-100,NaN
3,3,▁to,1.0,0,O
4,4,▁issue,2.0,0,O
5,5,▁6,3.0,0,O
6,6,▁.,4.0,0,O
7,7,▁4,5.0,0,O
8,8,▁crore,6.0,0,O
9,9,▁warrants,7.0,0,O


## Step 9. Build a minimal ATE batch collator and run a smoke test

This verifies that multiple ATE examples can be padded into a single batch without shape mismatches.

In [17]:
def collate_ate_batch(examples, tokenizer, max_length=128):
    encodings = [encode_ate_example(ex, tokenizer, max_length=max_length) for ex in examples]

    # Keep debug-only fields such as offset_mapping out of tokenizer.pad,
    # otherwise Hugging Face will try to tensorize nested span tuples.
    pad_ready_encodings = []
    for enc in encodings:
        enc_for_pad = dict(enc)
        enc_for_pad.pop("offset_mapping", None)
        enc_for_pad.pop("labels", None)
        pad_ready_encodings.append(enc_for_pad)

    batch = tokenizer.pad(
        pad_ready_encodings,
        padding=True,
        return_tensors="pt"
    )

    max_len = batch["input_ids"].shape[1]
    padded_labels = []
    for enc in encodings:
        labels = enc["labels"]
        padded = labels + [IGNORE_INDEX] * (max_len - len(labels))
        padded_labels.append(padded)

    batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
    return batch

In [18]:
ate_batch = collate_ate_batch(ate_train[:4], tokenizer, max_length=MAX_LEN_ATE)

print("ATE batch input_ids:", ate_batch["input_ids"].shape)
print("ATE batch attention_mask:", ate_batch["attention_mask"].shape)
print("ATE batch labels:", ate_batch["labels"].shape)

ATE batch input_ids: torch.Size([4, 14])
ATE batch attention_mask: torch.Size([4, 14])
ATE batch labels: torch.Size([4, 14])


## Step 10. Save ATE smoke validation results

In [19]:
ate_smoke_report = {
    "backbone": BACKBONE_NAME,
    "train_examples": len(ate_train),
    "val_examples": len(ate_val),
    "test_examples": len(ate_test),
    "sample_word_len": len(ate_sample["tokens"]),
    "sample_encoded_len": len(ate_encoded["input_ids"]),
    "batch_shape_input_ids": list(ate_batch["input_ids"].shape),
    "batch_shape_attention_mask": list(ate_batch["attention_mask"].shape),
    "batch_shape_labels": list(ate_batch["labels"].shape),
    "first_sample_token_label_len_match": len(ate_sample["tokens"]) == len(ate_sample["bio_labels"])
}

with open(ATE_SMOKE_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(ate_smoke_report, f, ensure_ascii=False, indent=2)

pprint(ate_smoke_report)

{'backbone': 'microsoft/deberta-v3-base',
 'batch_shape_attention_mask': [4, 14],
 'batch_shape_input_ids': [4, 14],
 'batch_shape_labels': [4, 14],
 'first_sample_token_label_len_match': True,
 'sample_encoded_len': 13,
 'sample_word_len': 10,
 'test_examples': 1079,
 'train_examples': 8576,
 'val_examples': 1071}


## Step 11. Inspect one ASC sample

ASC is one record per (sentence, aspect, sentiment) pair.
The core model-facing task is to turn sentence-aspect pairs into tokenized classification inputs.

In [20]:
asc_sample = asc_train[0]
pprint(asc_sample)

{'aspect': 'SpiceJet',
 'char_from': 0,
 'char_to': 8,
 'doc_id': 'sentfin_000001',
 'input_text_template': 'SpiceJet to issue 6.4 crore warrants to promoters '
                        '[SEP] SpiceJet',
 'match_type': 'whole_word_unique',
 'raw_id': 1,
 'sentence': 'SpiceJet to issue 6.4 crore warrants to promoters',
 'sentiment': 'neutral',
 'split': 'train'}


In [21]:
print("Sentence:", asc_sample["sentence"])
print("Aspect:", asc_sample["aspect"])
print("Sentiment:", asc_sample["sentiment"])

Sentence: SpiceJet to issue 6.4 crore warrants to promoters
Aspect: SpiceJet
Sentiment: neutral


## Step 12. Define ASC input construction

The proposal uses a sentence-plus-aspect format:
[CLS] sentence [SEP] aspect [SEP]

In Hugging Face tokenizers, we naturally implement this as a text pair:
tokenizer(sentence, aspect, ...)

In [22]:
def encode_asc_example(example, tokenizer, max_length=128):
    encoding = tokenizer(
        example["sentence"],
        example["aspect"],
        truncation=True,
        max_length=max_length,
        padding=False
    )
    encoding["label"] = ASC_LABEL2ID[example["sentiment"]]
    return encoding

In [23]:
asc_encoded = encode_asc_example(asc_sample, tokenizer, max_length=MAX_LEN_ASC)

print("input_ids length:", len(asc_encoded["input_ids"]))
print("attention_mask length:", len(asc_encoded["attention_mask"]))
print("label:", asc_encoded["label"])
print("tokens:", tokenizer.convert_ids_to_tokens(asc_encoded["input_ids"])[:40])

input_ids length: 16
attention_mask length: 16
label: 1
tokens: ['[CLS]', '▁Spice', 'Jet', '▁to', '▁issue', '▁6', '.', '4', '▁crore', '▁warrants', '▁to', '▁promoters', '[SEP]', '▁Spice', 'Jet', '[SEP]']


## Step 13. Sanity-check ASC tokenization

We verify that the aspect is truly included in the tokenized input and that the label mapping is valid.

In [24]:
tokens = tokenizer.convert_ids_to_tokens(asc_encoded["input_ids"])
pd.DataFrame({
    "idx": list(range(len(tokens))),
    "token": tokens
}).head(60)

,idx,token
0,0,[CLS]
1,1,▁Spice
2,2,Jet
3,3,▁to
4,4,▁issue
5,5,▁6
6,6,.
7,7,4
8,8,▁crore
9,9,▁warrants


In [25]:
assert asc_sample["sentiment"] in ASC_LABEL2ID
print("ASC label id:", ASC_LABEL2ID[asc_sample["sentiment"]])

ASC label id: 1


## Step 14. Build a minimal ASC batch collator and run a smoke test

In [26]:
def collate_asc_batch(examples, tokenizer, max_length=128):
    encodings = [encode_asc_example(ex, tokenizer, max_length=max_length) for ex in examples]

    labels = [enc.pop("label") for enc in encodings]

    batch = tokenizer.pad(
        encodings,
        padding=True,
        return_tensors="pt"
    )
    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch

In [27]:
asc_batch = collate_asc_batch(asc_train[:4], tokenizer, max_length=MAX_LEN_ASC)

print("ASC batch input_ids:", asc_batch["input_ids"].shape)
print("ASC batch attention_mask:", asc_batch["attention_mask"].shape)
print("ASC batch labels:", asc_batch["labels"].shape)
print("ASC batch label values:", asc_batch["labels"])

ASC batch input_ids: torch.Size([4, 19])
ASC batch attention_mask: torch.Size([4, 19])
ASC batch labels: torch.Size([4])
ASC batch label values: tensor([1, 1, 2, 2])


## Step 15. Save ASC smoke validation results

In [28]:
asc_smoke_report = {
    "backbone": BACKBONE_NAME,
    "train_examples": len(asc_train),
    "val_examples": len(asc_val),
    "test_examples": len(asc_test),
    "sample_encoded_len": len(asc_encoded["input_ids"]),
    "sample_label_id": int(asc_encoded["label"]),
    "batch_shape_input_ids": list(asc_batch["input_ids"].shape),
    "batch_shape_attention_mask": list(asc_batch["attention_mask"].shape),
    "batch_shape_labels": list(asc_batch["labels"].shape),
    "all_train_labels_valid": all(rec["sentiment"] in ASC_LABEL2ID for rec in asc_train)
}

with open(ASC_SMOKE_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(asc_smoke_report, f, ensure_ascii=False, indent=2)

pprint(asc_smoke_report)

{'all_train_labels_valid': True,
 'backbone': 'microsoft/deberta-v3-base',
 'batch_shape_attention_mask': [4, 19],
 'batch_shape_input_ids': [4, 19],
 'batch_shape_labels': [4],
 'sample_encoded_len': 16,
 'sample_label_id': 1,
 'test_examples': 1468,
 'train_examples': 11443,
 'val_examples': 1438}


## Step 16. Define shared prediction schemas

This is the most important interface contract for downstream integration.

We define:
1. ATE prediction schema
2. ASC prediction schema
3. merged end-to-end ABSA schema

In [29]:
ATE_PREDICTION_SCHEMA = {
    "doc_id": "sentfin_xxxxxx",
    "predicted_spans": [
        {
            "term": "example aspect",
            "char_from": 0,
            "char_to": 14
        }
    ]
}

ASC_PREDICTION_SCHEMA = {
    "doc_id": "sentfin_xxxxxx",
    "aspect_predictions": [
        {
            "term": "example aspect",
            "char_from": 0,
            "char_to": 14,
            "sentiment": "positive"
        }
    ]
}

MERGED_ABSA_SCHEMA = {
    "doc_id": "sentfin_xxxxxx",
    "predicted_pairs": [
        {
            "term": "example aspect",
            "char_from": 0,
            "char_to": 14,
            "sentiment": "positive"
        }
    ]
}

print("ATE schema:")
pprint(ATE_PREDICTION_SCHEMA)
print("\nASC schema:")
pprint(ASC_PREDICTION_SCHEMA)
print("\nMerged schema:")
pprint(MERGED_ABSA_SCHEMA)

ATE schema:
{'doc_id': 'sentfin_xxxxxx',
 'predicted_spans': [{'char_from': 0, 'char_to': 14, 'term': 'example aspect'}]}

ASC schema:
{'aspect_predictions': [{'char_from': 0,
                         'char_to': 14,
                         'sentiment': 'positive',
                         'term': 'example aspect'}],
 'doc_id': 'sentfin_xxxxxx'}

Merged schema:
{'doc_id': 'sentfin_xxxxxx',
 'predicted_pairs': [{'char_from': 0,
                      'char_to': 14,
                      'sentiment': 'positive',
                      'term': 'example aspect'}]}


## Step 17. Inspect one canonical headline record for merge logic

The merge branch should not invent its own input source.
It must rely on the same canonical headline-level records used by all branches.

In [30]:
canon_sample = canonical_train[0]
pprint(canon_sample)

{'all_aspects_matched': True,
 'aspects': [{'char_from': 0,
              'char_to': 8,
              'match_status': 'matched',
              'match_type': 'whole_word_unique',
              'sentiment': 'neutral',
              'term': 'SpiceJet'}],
 'doc_id': 'sentfin_000001',
 'group_key': 'spicejet to issue 64 crore warrants to promoters',
 'label_signature': 'neutral',
 'num_aspects': 1,
 'num_matched_aspects': 1,
 'num_review_aspects': 0,
 'raw_id': 1,
 'split': 'train',
 'text': 'SpiceJet to issue 6.4 crore warrants to promoters'}


In [31]:
print("Canonical text:", canon_sample["text"])
print("Canonical aspects:")
for asp in canon_sample["aspects"]:
    print(asp)

Canonical text: SpiceJet to issue 6.4 crore warrants to promoters
Canonical aspects:
{'term': 'SpiceJet', 'sentiment': 'neutral', 'char_from': 0, 'char_to': 8, 'match_status': 'matched', 'match_type': 'whole_word_unique'}


## Step 18. Simulate merge inputs using gold or pseudo predictions

This smoke test does NOT train the joint model.
It only verifies that ATE-style outputs and ASC-style outputs can be combined into one unified schema.

In [32]:
def build_fake_ate_prediction_from_gold(canonical_record):
    spans = []
    for asp in canonical_record["aspects"]:
        if asp["match_status"] == "matched":
            spans.append({
                "term": asp["term"],
                "char_from": asp["char_from"],
                "char_to": asp["char_to"]
            })
    return {
        "doc_id": canonical_record["doc_id"],
        "predicted_spans": spans
    }

def build_fake_asc_prediction_from_gold(canonical_record):
    preds = []
    for asp in canonical_record["aspects"]:
        if asp["match_status"] == "matched":
            preds.append({
                "term": asp["term"],
                "char_from": asp["char_from"],
                "char_to": asp["char_to"],
                "sentiment": asp["sentiment"]
            })
    return {
        "doc_id": canonical_record["doc_id"],
        "aspect_predictions": preds
    }

In [33]:
fake_ate_pred = build_fake_ate_prediction_from_gold(canon_sample)
fake_asc_pred = build_fake_asc_prediction_from_gold(canon_sample)

print("Fake ATE prediction:")
pprint(fake_ate_pred)
print("\nFake ASC prediction:")
pprint(fake_asc_pred)

Fake ATE prediction:
{'doc_id': 'sentfin_000001',
 'predicted_spans': [{'char_from': 0, 'char_to': 8, 'term': 'SpiceJet'}]}

Fake ASC prediction:
{'aspect_predictions': [{'char_from': 0,
                         'char_to': 8,
                         'sentiment': 'neutral',
                         'term': 'SpiceJet'}],
 'doc_id': 'sentfin_000001'}


## Step 19. Merge ATE-style and ASC-style outputs into end-to-end ABSA format

This validates the schema expected by later pipeline evaluation.

In [34]:
def merge_ate_and_asc_predictions(ate_pred, asc_pred):
    ate_spans = {
        (p["term"], p["char_from"], p["char_to"]): p
        for p in ate_pred["predicted_spans"]
    }

    merged_pairs = []
    for p in asc_pred["aspect_predictions"]:
        key = (p["term"], p["char_from"], p["char_to"])
        if key in ate_spans:
            merged_pairs.append({
                "term": p["term"],
                "char_from": p["char_from"],
                "char_to": p["char_to"],
                "sentiment": p["sentiment"]
            })

    return {
        "doc_id": ate_pred["doc_id"],
        "predicted_pairs": merged_pairs
    }

In [35]:
merged_pred = merge_ate_and_asc_predictions(fake_ate_pred, fake_asc_pred)
pprint(merged_pred)

{'doc_id': 'sentfin_000001',
 'predicted_pairs': [{'char_from': 0,
                      'char_to': 8,
                      'sentiment': 'neutral',
                      'term': 'SpiceJet'}]}


## Step 20. Validate merge compatibility

We verify that:
- document IDs match,
- merged pairs are structurally valid,
- each merged pair has the required keys.

In [36]:
def validate_merged_schema(pred):
    if "doc_id" not in pred or "predicted_pairs" not in pred:
        return False
    for pair in pred["predicted_pairs"]:
        required = {"term", "char_from", "char_to", "sentiment"}
        if not required.issubset(set(pair.keys())):
            return False
    return True

merge_valid = validate_merged_schema(merged_pred)
print("Merge schema valid:", merge_valid)

Merge schema valid: True


In [37]:
merge_smoke_report = {
    "canonical_train_rows": len(canonical_train),
    "sample_doc_id": canon_sample["doc_id"],
    "fake_ate_pred_count": len(fake_ate_pred["predicted_spans"]),
    "fake_asc_pred_count": len(fake_asc_pred["aspect_predictions"]),
    "merged_pair_count": len(merged_pred["predicted_pairs"]),
    "merge_schema_valid": bool(merge_valid)
}

with open(MERGE_SMOKE_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(merge_smoke_report, f, ensure_ascii=False, indent=2)

pprint(merge_smoke_report)

{'canonical_train_rows': 8576,
 'fake_asc_pred_count': 1,
 'fake_ate_pred_count': 1,
 'merge_schema_valid': True,
 'merged_pair_count': 1,
 'sample_doc_id': 'sentfin_000001'}


## Step 21. Evaluate whether Phase 3 is complete

This phase is complete if:
- the tokenizer loads successfully,
- ATE examples can be tokenized and aligned,
- ASC examples can be tokenized as sentence-aspect pairs,
- batch collation works for both branches,
- the shared prediction schema is defined,
- merge smoke validation passes.

In [38]:
checklist = [
    ("Tokenizer loaded", tokenizer is not None),
    ("ATE files loaded", len(ate_train) > 0 and len(ate_val) > 0 and len(ate_test) > 0),
    ("ASC files loaded", len(asc_train) > 0 and len(asc_val) > 0 and len(asc_test) > 0),
    ("Canonical files loaded", len(canonical_train) > 0 and len(canonical_val) > 0 and len(canonical_test) > 0),
    ("ATE fields valid", ate_ok),
    ("ASC fields valid", asc_ok),
    ("Canonical fields valid", canon_ok),
    ("ATE alignment produced labels", "labels" in ate_encoded and len(ate_encoded["labels"]) == len(ate_encoded["input_ids"])),
    ("ATE batch built", tuple(ate_batch["input_ids"].shape)[0] == 4),
    ("ASC encoding produced label", "label" in asc_encoded),
    ("ASC batch built", tuple(asc_batch["input_ids"].shape)[0] == 4),
    ("Shared prediction schemas defined", True),
    ("Merge smoke validation passed", merge_valid),
    ("ATE smoke report saved", ATE_SMOKE_REPORT_PATH.exists()),
    ("ASC smoke report saved", ASC_SMOKE_REPORT_PATH.exists()),
    ("Merge smoke report saved", MERGE_SMOKE_REPORT_PATH.exists()),
]

df_checklist = pd.DataFrame(checklist, columns=["check_item", "passed"])
df_checklist.to_csv(PHASE3_CHECKLIST_PATH, index=False)

display(df_checklist)
print("PHASE 3 COMPLETE:", df_checklist["passed"].all())

,check_item,passed
0,Tokenizer loaded,True
1,ATE files loaded,True
2,ASC files loaded,True
3,Canonical files loaded,True
4,ATE fields valid,True
5,ASC fields valid,True
6,Canonical fields valid,True
7,ATE alignment produced labels,True
8,ATE batch built,True
9,ASC encoding produced label,True


PHASE 3 COMPLETE: True


## Phase 3 conclusion

This phase is successful if the finance ABSA data can be loaded into a shared DeBERTa-based preprocessing pipeline and if all three project branches are interface-compatible:

1. ATE can produce tokenized sequence-labeling inputs with aligned labels;
2. ASC can produce tokenized sentence-aspect classification inputs;
3. merge/pipeline integration can consume standardized prediction schemas.

The next phase will perform minimal training validation:

- one small ATE training loop,
- one small ASC training loop,
- one minimal merge / pipeline validation,
- without any advanced tuning or ablation yet.